# Modelo de Predicción Player + Team Attributes con Rachas y Días de Descanso

### Arquitectura del Pipeline de Ingeniería de Features y Modelado XGBoost

Este notebook consolida un pipeline end-to-end orientado a la predicción del resultado de partidos de fútbol (Target: `Home Win (H)`, `Draw (D)`, `Away Win (A)`). La arquitectura extiende los atributos estáticos de jugadores y equipos mediante la inyección de dos bloques de features macro-dinámicas:
1. **Rachas Recientes (Streaks):** Ventanas móviles de 3, 5 y 10 partidos para capturar la inercia competitiva.
2. **Días de Descanso Efectivo (Rest Days):** Ventanas de recuperación biológica y fatiga acumulada (`home_rest_days`, `away_rest_days`, `rest_days_diff`), calculadas mediante la unificación cronológica de fixtures para mitigar omisiones de calendario.

Se previene estrictamente el *data leakage* temporal ordenando secuencialmente el dataset y aplicando desplazamientos (`shift`) antes de la fase de entrenamiento con un clasificador basado en Gradient Boosting (`XGBClassifier`).

In [1]:
# =============================================================================
# 1. DEPENDENCIAS Y CONFIGURACIÓN DEL ENTORNO
# =============================================================================
import sqlite3 as sql
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

In [2]:
# =============================================================================
# 2. INGESTIÓN DE DATOS DESDE REPOSITORIO SQLITE
# =============================================================================
conn = sql.connect('../../data/database.sqlite')

# Extracción de las tablas base
df_match = pd.read_sql_query("SELECT * FROM Match", conn)
df_team_attrs = pd.read_sql_query("SELECT * FROM Team_Attributes", conn)
df_player_attrs = pd.read_sql_query("SELECT * FROM Player_Attributes", conn)

conn.close()

# Cast estructural de vectores temporales
df_match['date'] = pd.to_datetime(df_match['date'])
df_team_attrs['date'] = pd.to_datetime(df_team_attrs['date'])
df_player_attrs['date'] = pd.to_datetime(df_player_attrs['date'])

print("Ingestión finalizada. Registros de partidos cargados:", df_match.shape[0])

Ingestión finalizada. Registros de partidos cargados: 25979


In [3]:
# =============================================================================
# 3. INGENIERÍA DE FEATURES: DÍAS DE DESCANSO EFECTIVO (REST DAYS)
# =============================================================================

# Isolation de las estructuras temporales de participación por condición
home_dates = df_match[['match_api_id', 'date', 'home_team_api_id']].copy()
home_dates.columns = ['match_api_id', 'date', 'team_api_id']

away_dates = df_match[['match_api_id', 'date', 'away_team_api_id']].copy()
away_dates.columns = ['match_api_id', 'date', 'team_api_id']

# Consolidación del histórico unificado cronológicamente
full_fixtures = pd.concat([home_dates, away_dates], axis=0).sort_values(by=['team_api_id', 'date'])

# Cálculo de la fecha del encuentro inmediato anterior por grupo indexado
full_fixtures['previous_match_date'] = full_fixtures.groupby('team_api_id')['date'].shift(1)

# Computación del delta temporal en días vectorizados
full_fixtures['rest_days'] = (
    (full_fixtures['date'] - full_fixtures['previous_match_date']).dt.total_seconds() / 86400.0
)

# Mapeo de los vectores de descanso al DataFrame maestro
df_match = df_match.merge(
    full_fixtures[['match_api_id', 'team_api_id', 'rest_days']],
    left_on=['match_api_id', 'home_team_api_id'],
    right_on=['match_api_id', 'team_api_id'],
    how='left'
).rename(columns={'rest_days': 'home_rest_days'}).drop(columns=['team_api_id'])

df_match = df_match.merge(
    full_fixtures[['match_api_id', 'team_api_id', 'rest_days']],
    left_on=['match_api_id', 'away_team_api_id'],
    right_on=['match_api_id', 'team_api_id'],
    how='left'
).rename(columns={'rest_days': 'away_rest_days'}).drop(columns=['team_api_id'])

# Gradiente diferencial neto de descanso
df_match['rest_days_diff'] = df_match['home_rest_days'] - df_match['away_rest_days']

rest_days_cols = ['home_rest_days', 'away_rest_days', 'rest_days_diff']
print("Features de Días de Descanso integradas con éxito.")

Features de Días de Descanso integradas con éxito.


In [4]:
# =============================================================================
# 4. INGENIERÍA DE FEATURES: CÁLCULO DE RACHAS RECIENTES (STREAKS)
# =============================================================================

# Ordenamiento cronológico imperativo absoluto antes de procesar rachas
df_match = df_match.sort_values(by='date').reset_index(drop=True)

def compute_points(row, team_type='home'):
    if row['home_team_goal'] == row['away_team_goal']:
        return 1
    if team_type == 'home' and row['home_team_goal'] > row['away_team_goal']:
        return 3
    if team_type == 'away' and row['away_team_goal'] > row['home_team_goal']:
        return 3
    return 0

df_match['home_points'] = df_match.apply(lambda r: compute_points(r, 'home'), axis=1)
df_match['away_points'] = df_match.apply(lambda r: compute_points(r, 'away'), axis=1)

# Construcción del log cronológico de puntos por equipo
team_history = {}
for idx, row in df_match.iterrows():
    h_team = row['home_team_api_id']
    a_team = row['away_team_api_id']
    
    if h_team not in team_history: team_history[h_team] = []
    if a_team not in team_history: team_history[a_team] = []
    
    team_history[h_team].append({'match_idx': idx, 'points': row['home_points']})
    team_history[a_team].append({'match_idx': idx, 'points': row['away_points']})

def get_past_streak(team_id, current_match_idx, window):
    matches = team_history.get(team_id, [])
    # Filtrar únicamente los partidos estrictamente anteriores del equipo para evitar leakage
    past_matches = [m['points'] for m in matches if m['match_idx'] < current_match_idx]
    if len(past_matches) == 0:
        return 0.0
    effective_window = min(len(past_matches), window)
    return sum(past_matches[-effective_window:]) / effective_window

windows = [3, 5, 10]
for w in windows:
    df_match[f'home_streak_{w}'] = df_match.apply(lambda r: get_past_streak(r['home_team_api_id'], r.name, w), axis=1)
    df_match[f'away_streak_{w}'] = df_match.apply(lambda r: get_past_streak(r['away_team_api_id'], r.name, w), axis=1)

streak_cols = [
    'home_streak_3', 'home_streak_5', 'home_streak_10',
    'away_streak_3', 'away_streak_5', 'away_streak_10'
]
print("Features de Rachas Recientes procesadas e integradas.")

Features de Rachas Recientes procesadas e integradas.


In [5]:
# =============================================================================
# 5. PIPELINE DE INTEGRACIÓN DE ATRIBUTOS ESTÁTICOS (PLAYERS & TEAMS)
# =============================================================================

# Obtención de la última firma de atributos estáticos mapeados
latest_team_attrs = df_team_attrs.sort_values('date').groupby('team_api_id').last().reset_index()
latest_player_attrs = df_player_attrs.sort_values('date').groupby('player_api_id').last().reset_index()

team_numeric_cols = latest_team_attrs.select_dtypes(include=[np.number]).columns.drop(['id', 'team_api_id', 'team_fifa_api_id']).tolist()
player_numeric_cols = latest_player_attrs.select_dtypes(include=[np.number]).columns.drop(['id', 'player_fifa_api_id', 'player_api_id']).tolist()

player_map = latest_player_attrs.set_index('player_api_id')[player_numeric_cols].to_dict('index')
team_map = latest_team_attrs.set_index('team_api_id')[team_numeric_cols].to_dict('index')

# Parseo adaptativo de alineaciones titulares
home_player_cols = [f'home_player_{i}' for i in range(1, 12)]
away_player_cols = [f'away_player_{i}' for i in range(1, 12)]

features_list = []
targets_list = []

for idx, row in df_match.iterrows():
    # Clasificación del Target ordinal
    if row['home_team_goal'] > row['away_team_goal']:
        target = 0 # Home Win
    elif row['home_team_goal'] == row['away_team_goal']:
        target = 1 # Draw
    else:
        target = 2 # Away Win
        
    match_feats = {}
    
    # Inyección de Atributos del Equipo Local y Visitante
    h_team_feats = team_map.get(row['home_team_api_id'], {c: np.nan for c in team_numeric_cols})
    a_team_feats = team_map.get(row['away_team_api_id'], {c: np.nan for c in team_numeric_cols})
    for c in team_numeric_cols:
        match_feats[f'home_team_{c}'] = h_team_feats[c]
        match_feats[f'away_team_{c}'] = a_team_feats[c]
        
    # Agregación por promedio de los Atributos de los Jugadores
    h_players_feats = [player_map.get(row[p_col], {}) for p_col in home_player_cols if not pd.isna(row[p_col])]
    a_players_feats = [player_map.get(row[p_col], {}) for p_col in away_player_cols if not pd.isna(row[p_col])]
    
    for c in player_numeric_cols:
        match_feats[f'home_players_mean_{c}'] = np.mean([p[c] for p in h_players_feats if c in p]) if h_players_feats else np.nan
        match_feats[f'away_players_mean_{c}'] = np.mean([p[c] for p in a_players_feats if c in p]) if a_players_feats else np.nan
        
    # Integración directa de Features Dinámicas Integradas previamente
    for col in streak_cols:
        match_feats[col] = row[col]
    for col in rest_days_cols:
        match_feats[col] = row[col]
        
    features_list.append(match_feats)
    targets_list.append(target)

df_features = pd.DataFrame(features_list)
y = np.array(targets_list)

print("Matriz de features estructurada de forma consolidada. Dimensión:", df_features.shape)

Matriz de features estructurada de forma consolidada. Dimensión: (25979, 97)


In [6]:
# =============================================================================
# 6. PARTIDAS DE ENTRENAMIENTO, VALIDACIÓN Y OPTIMIZACIÓN XGBOOST
# =============================================================================

# Tratamiento de valores ausentes estructurales previo al entrenamiento
df_features = df_features.fillna(df_features.mean())

# Split de validación asumiendo aleatoriedad controlada por estado interno
X_train, X_test, y_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=42, stratify=y
)

# Instanciación y parametrización del modelo XGBoost Multiclase
model = XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

print("Iniciando fase de ajuste del estimador XGBClassifier...")
model.fit(X_train, y_train)

# Inferencia de la matriz de validación
y_pred = model.predict(X_test)

# Métricas de Performance de la Red
accuracy = accuracy_score(y_test, y_pred)
print(f"\n[RESULTADO] Accuracy Global del Sistema: {accuracy:.4f}\n")
print("=== Reporte de Clasificación Estructural ===")
print(classification_report(y_test, y_pred, target_names=['Home Win (0)', 'Draw (1)', 'Away Win (2)']))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

Iniciando fase de ajuste del estimador XGBClassifier...

[RESULTADO] Accuracy Global del Sistema: 0.5191

=== Reporte de Clasificación Estructural ===
              precision    recall  f1-score   support

Home Win (0)       0.54      0.83      0.65      2384
    Draw (1)       0.34      0.03      0.06      1319
Away Win (2)       0.49      0.46      0.47      1493

    accuracy                           0.52      5196
   macro avg       0.45      0.44      0.39      5196
weighted avg       0.47      0.52      0.45      5196

=== Matriz de Confusión ===
[[1972   50  362]
 [ 928   43  348]
 [ 777   34  682]]


In [7]:
# =============================================================================
# 7. ANÁLISIS DE IMPORTANCIA DE FEATURES (INFLUENCIA DE REST DAYS Y STREAKS)
# =============================================================================
importances = model.feature_importances_
indices = np.argsort(importances)[::-1]

print("Top 15 Features con mayor ganancia de información teórica en el modelo:")
for f in range(min(15, df_features.shape[1])):
    print(f"{f + 1}. Feature: {df_features.columns[indices[f]]} ({importances[indices[f]]*100:.2f}%)")

Top 15 Features con mayor ganancia de información teórica en el modelo:
1. Feature: home_streak_10 (3.64%)
2. Feature: home_players_mean_overall_rating (3.62%)
3. Feature: away_players_mean_overall_rating (3.53%)
4. Feature: away_streak_10 (3.16%)
5. Feature: home_players_mean_ball_control (2.63%)
6. Feature: away_players_mean_ball_control (2.16%)
7. Feature: away_players_mean_potential (2.00%)
8. Feature: home_players_mean_reactions (1.97%)
9. Feature: away_players_mean_dribbling (1.85%)
10. Feature: home_players_mean_short_passing (1.82%)
11. Feature: away_players_mean_short_passing (1.57%)
12. Feature: away_players_mean_vision (1.49%)
13. Feature: away_players_mean_reactions (1.31%)
14. Feature: home_players_mean_volleys (1.17%)
15. Feature: home_players_mean_positioning (1.15%)
